<a href="https://colab.research.google.com/github/nishithy13/ds2002-fa26/blob/main/notebooks/01-foundations/2026-09-09%20%E2%80%94%20SQLite%20Joins%20and%20Grouping%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQLite Joins and Grouping

**Studio — 2026-09-09 · Fall 2026**  
**Class time:** 45 minutes

---

## Connecting the tables

Monday every query hit one table. Today we answer questions that no single table can: *which artist gets played most?* lives across all three.

A join says "for each row here, find the matching rows there" and the `ON` clause is where you say what matching means. Get the `ON` wrong and you do not get an error — you get a wrong answer, sometimes a very confident-looking one. So we are going to break a join on purpose before we rely on one.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Worked example — plays, with the track titles attached

`plays` only stores a `track_id`. Nobody wants to read a report of numbers, so join to `tracks` to get the title. The letters `p` and `t` are aliases — they save typing and make it obvious which table each column came from.

In [2]:
q('''
SELECT p.play_id, p.user, t.title, t.genre
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
ORDER BY p.play_id
LIMIT 6
''')

,play_id,user,title,genre
0,100,ava,Skyline,Pop
1,101,ben,Skyline,Pop
2,102,ava,Aurora,Electronic
3,103,cara,Aurora,Electronic
4,104,ben,Nightfall,Electronic
5,105,ava,Foothills,Folk


### Worked example — when the join key is wrong

Here is the same query with one plausible-looking mistake: joining `plays.track_id` to `tracks.artist_id`. Both columns are integers, both exist, and SQLite runs it happily.

In [3]:
wrong = q('''
SELECT p.play_id, t.title
FROM plays p
JOIN tracks t ON p.track_id = t.artist_id   -- wrong column
''')
right = q('''
SELECT p.play_id, t.title
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
''')
print('wrong key ->', len(wrong), 'rows')
print('right key ->', len(right), 'rows')
wrong.head()

wrong key -> 0 rows
right key -> 11 rows


,play_id,title


Zero rows, no error. That is the good case — the bad case is when a wrong key *matches* and you get inflated counts.

**The habit:** after every join, check the row count against what you expected. There are 11 plays, so a join from `plays` to `tracks` should return 11 rows. Not 0, not 40.

This is the single most useful validation in this course, and it comes back on the midterm when you join sales to weather.

### Worked example — INNER vs LEFT

A plain `JOIN` is an **inner** join: rows survive only if they match on both sides. Two tracks in this database have never been played, so an inner join from `tracks` to `plays` silently drops them.

In [4]:
inner = q('''
SELECT t.track_id, t.title, COUNT(p.play_id) AS plays
FROM tracks t
JOIN plays p ON t.track_id = p.track_id
GROUP BY t.track_id, t.title
''')
left = q('''
SELECT t.track_id, t.title, COUNT(p.play_id) AS plays
FROM tracks t
LEFT JOIN plays p ON t.track_id = p.track_id
GROUP BY t.track_id, t.title
''')
print('INNER ->', len(inner), 'tracks | LEFT ->', len(left), 'tracks')
left

INNER -> 7 tracks | LEFT -> 9 tracks


,track_id,title,plays
0,10,Skyline,3
1,11,Undertow,1
2,12,Foothills,1
3,13,Aurora,3
4,14,Nightfall,1
5,15,Sol,1
6,16,Coastline,1
7,17,Ridgeline,0
8,18,Untitled Demo,0


The `LEFT JOIN` keeps every track and reports `0` for the ones nobody played. Which one you want depends entirely on the question:

- "What are people listening to?" — inner join is fine.
- "Which tracks are we wasting catalog space on?" — you need the zeros, so left join.

Note that `COUNT(p.play_id)` gives 0 for unmatched rows, while `COUNT(*)` would give 1, because there is still one row there. Count the column, not the row.

### Build 1 — the most-played artist

Chain both joins: `plays` -> `tracks` -> `artists`. Count plays per artist name, busiest first.

**Expected:** 4 artists, and the totals should add up to 11.

In [6]:
# TODO: SELECT a.name, COUNT(*) AS plays
#       FROM plays p JOIN tracks t ON ... JOIN artists a ON ...
#       GROUP BY a.name ORDER BY plays DESC
q('''
SELECT a.name, COUNT(*) AS plays
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
JOIN artists a ON t.artist_id = a.artist_id
GROUP BY a.name
ORDER BY plays DESC
''')

,name,plays
0,Nova Waves,4
1,Kestrel,4
2,The Blue Ridge,2
3,Marisol,1


In [9]:
# Validate: does your result account for all 11 plays?
result =  q('''
SELECT a.name, COUNT(*) AS plays
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
JOIN artists a ON t.artist_id = a.artist_id
GROUP BY a.name
ORDER BY plays DESC
''')
# TODO: sum the plays column of your result and compare to 11
result['plays'].sum()

np.int64(11)

### Build 2 — plays per genre

Same idea, grouped by genre instead. Watch what happens to the untagged track — should it appear? Say why in a comment.

In [10]:
q('''
SELECT t.genre, COUNT(*) AS plays
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
GROUP BY t.genre
ORDER BY plays DESC
''')

,genre,plays
0,Pop,4
1,Electronic,4
2,Folk,2
3,Latin,1


# The untagged track does not appear because it never got played.

### Build 3 — the artists worth promoting

Artists with more than two total plays. This needs `HAVING`, not `WHERE`.

In [11]:
q('''
SELECT a.name, COUNT(*) AS plays
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
JOIN artists a ON t.artist_id = a.artist_id
GROUP BY a.name
HAVING COUNT(*) > 2
ORDER BY plays DESC
''')

,name,plays
0,Nova Waves,4
1,Kestrel,4


### Build 4 — every artist, including the quiet ones

Now the harder version: list **all four** artists with their play counts, including any artist whose tracks nobody has played. Think about which table has to be on the left.

In [14]:
q('''
SELECT a.name, COUNT(p.play_id) AS plays
FROM artists a
LEFT JOIN tracks t ON a.artist_id = t.artist_id
LEFT JOIN plays p ON t.track_id = p.track_id
GROUP BY a.name
ORDER BY plays DESC
''')

,name,plays
0,Nova Waves,4
1,Kestrel,4
2,The Blue Ridge,2
3,Marisol,1


### Build 5 — total listening time per artist

Plays are not equal — a five-minute track is more listening than a three-minute one. Sum actual seconds listened per artist and convert to minutes.

In [20]:
q('''
SELECT a.name, SUM(t.seconds) / 60 AS minutes
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
JOIN artists a ON t.artist_id = a.artist_id
GROUP BY a.name
ORDER BY minutes DESC
''')

,name,minutes
0,Kestrel,19
1,Nova Waves,14
2,The Blue Ridge,6
3,Marisol,3


---

## Checkpoint (participation)

Report your most-played artist and whether your play counts added up to 11.

Work the last few minutes in groups of 4–5, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Wednesday 11:59pm ET**. One submission per person, not per group.

In [21]:
# Checkpoint
top_artist = 'Nova Waves'        # TODO: from Build 1
plays_accounted = 11     # TODO: the sum of your plays column
inner_vs_left = 'I should use a LEFT JOIN when I want to keep every row from the left table, despite having no matches.'     # TODO: one sentence on when you need LEFT JOIN

print('top artist:', top_artist)
print('plays accounted for:', plays_accounted, 'of 11')
print('when I need a LEFT JOIN:', inner_vs_left)

top artist: Nova Waves
plays accounted for: 11 of 11
when I need a LEFT JOIN: I should use a LEFT JOIN when I want to keep every row from the left table, despite having no matches.
